# 04 – Who should we contact?

Not every user who leaves a cart is the same. Here the abandoners are compared with the buyers,
to find the group with the best chance to come back after a reminder.
Data: dbt tables `otto.mart_sessions` and `otto.mart_products`.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine

sns.set_theme(style="white", font_scale=1.1)

DATA_DIR = Path("../data/processed")
FIG_DIR = Path("../docs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# same colours as in notebook 03
COLORS = {
    "abandoner": "#e4572e",
    "quick_buyer": "#2a9d8f",
    "same_day_buyer": "#8ab17d",
    "slow_buyer": "#264653",
}
LABELS = {
    "abandoner": "Never ordered",
    "quick_buyer": "Within 1 hour",
    "same_day_buyer": "1–24 hours",
    "slow_buyer": "After 1+ day",
}
TYPE_ORDER = ["abandoner", "quick_buyer", "same_day_buyer", "slow_buyer"]

## Load the data

Downloaded once from PostgreSQL, then read from the local files.

In [ ]:
load_dotenv("../.env", override=True)
engine = create_engine(
    f"postgresql://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASS')}"
    f"@{os.getenv('POSTGRES_HOST')}:{os.getenv('POSTGRES_PORT')}/{os.getenv('POSTGRES_DB')}"
)

def load_table(table_name, file_name):
    file_path = DATA_DIR / file_name
    if file_path.exists():
        return pd.read_parquet(file_path)
    table = pd.read_sql(f"SELECT * FROM otto.{table_name}", engine)
    table.to_parquet(file_path, index=False)
    return table

sessions = load_table("mart_sessions", "mart_sessions_all.parquet")
products = load_table("mart_products", "mart_products.parquet")
print(sessions.shape, products.shape)

## 1. How do abandoners behave compared with buyers?

Only users with a cart (no `late_cart` or `other`). "Came back" means active on 2 or more different days.

In [ ]:
cart_users = sessions[sessions["buyer_type"].isin(TYPE_ORDER)].copy()
cart_users["came_back"] = cart_users["active_days"] >= 2

behaviour = cart_users.groupby("buyer_type").agg(
    users=("session_id", "count"),
    median_clicks=("n_clicks", "median"),
    median_products=("n_products", "median"),
    median_cart_adds=("n_carts", "median"),
    came_back_share=("came_back", "mean"),
)
behaviour = behaviour.reindex(TYPE_ORDER)
behaviour["came_back_share"] = (behaviour["came_back_share"] * 100).round(1)
behaviour

In [ ]:
plot_data = behaviour.reset_index()
plot_data["label"] = plot_data["buyer_type"].map(LABELS)
abandoner_share = behaviour.loc["abandoner", "came_back_share"]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(plot_data["label"], plot_data["came_back_share"], color=[COLORS[t] for t in plot_data["buyer_type"]])
ax.bar_label(bars, fmt="%.0f%%", padding=3)

ax.set_title(f"{abandoner_share:.0f}% of abandoners were active on more than one day", loc="left", fontweight="bold")
ax.set_ylabel("% active on 2+ days")
ax.set_xlabel("Buyer type (when they ordered after the cart)")
ax.set_ylim(0, 110)
sns.despine()
plt.tight_layout()
plt.savefig(FIG_DIR / "05_came_back_by_type.png", dpi=200)
plt.show()

## 2. The target group

Abandoners who came back on another day but still didn't buy. They showed interest more than once,
like the slow buyers who did order in the end, so a reminder has the best chance with them.

In [ ]:
abandoners = cart_users[cart_users["buyer_type"] == "abandoner"]
returning = abandoners[abandoners["came_back"]]
one_visit = abandoners[~abandoners["came_back"]]
slow_buyers = cart_users[cart_users["buyer_type"] == "slow_buyer"]

print(f"Abandoners:                {len(abandoners):,}")
print(f"Came back on another day:  {len(returning):,} ({len(returning) / len(abandoners):.0%})")
print(f"Only one day:              {len(one_visit):,} ({len(one_visit) / len(abandoners):.0%})")

In [ ]:
columns = ["n_clicks", "n_products", "n_carts", "active_days"]

compare = pd.DataFrame({
    "One-day abandoners": one_visit[columns].median(),
    "Returning abandoners": returning[columns].median(),
    "Slow buyers": slow_buyers[columns].median(),
})
compare

In [ ]:
groups = pd.DataFrame({
    "group": ["Only one day", "Came back on another day"],
    "users": [len(one_visit), len(returning)],
})
groups["share"] = groups["users"] / groups["users"].sum() * 100

fig, ax = plt.subplots(figsize=(8, 3.5))
bars = ax.barh(groups["group"], groups["users"], color=["#c9ced6", COLORS["abandoner"]])
for bar, users, share in zip(bars, groups["users"], groups["share"]):
    ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2, f"{users:,} ({share:.0f}%)", va="center")

ax.set_title(f"{len(returning):,} abandoners came back but still didn't buy", loc="left", fontweight="bold")
ax.set_xlabel("Abandoners (users with a cart but no order)")
ax.set_xlim(0, groups["users"].max() * 1.3)
sns.despine()
plt.tight_layout()
plt.savefig(FIG_DIR / "06_target_group.png", dpi=200)
plt.show()

## 3. Products that are often left in the cart (bonus)

Only products added to the cart at least 50 times, so the rate is not based on a few users.

In [ ]:
overall_rate = products["n_orders"].sum() / products["n_carts"].sum()
print(f"Average cart-to-order rate: {overall_rate:.1%}")

often_in_cart = products[products["n_carts"] >= 50]
hesitation = often_in_cart.sort_values("cart_to_order_rate").head(10)
hesitation